# LLM Quantization Toolkit

This notebook implements and compares multiple quantization methods for Large Language Models:

- **RTN (Round-To-Nearest)**: Simple weight-only quantization with group-wise scaling
- **GPTQ**: Advanced quantization with sequential layer-wise optimization  
- **AWQ (Activation-aware Weight Quantization)**: Preserves important weights based on activation magnitudes

## Target Model
- **Model**: OpenLLaMA 7B
- **Output**: Quantized models at 4-bit and 8-bit precision
- **Calibration**: WikiText-2 dataset (128 samples)

## Use Cases
- Mobile deployment (reduced memory footprint)
- Edge devices (lower bandwidth requirements)
- Cost-effective inference (reduced compute)

## 1. Imports and Configuration

In [1]:
# ============================================================================
# IMPORTS AND CONFIGURATION
# ============================================================================

# Standard library imports
from pathlib import Path
from typing import Iterable, List

# PyTorch and deep learning
import torch
import torch.nn as nn

# Hugging Face transformers and datasets
from transformers import AutoModelForCausalLM, AutoTokenizer
from datasets import load_dataset

# Project imports
import utils.model_loader as model_loader
import utils.dataloader as dataloader

# ============================================================================
# CONFIGURATION
# ============================================================================

# Model configuration
MODEL_ID = "openlm-research/open_llama_7b"  # Target model to quantize

# Output configuration  
OUTPUT_ROOT = Path("./quantized_models")  # Directory for saving quantized models

# Calibration configuration
MIN_CALIB_SAMPLES = 128  # Minimum number of calibration samples for quantization

In [2]:
# Check if cache is on the mounted drive
from huggingface_hub.constants import HF_HUB_CACHE
print(f"Hugging Face Hub is pointing to: {HF_HUB_CACHE}")

Hugging Face Hub is pointing to: /mnt/huggingface_cache/hub


## 2. Calibration Data Loading

Calibration data is crucial for quantization algorithms (especially GPTQ and AWQ) to determine optimal quantization parameters. We use WikiText-2, a standard language modeling dataset with diverse, high-quality text.

In [3]:
# Load calibration data (this runs when the cell executes)
CALIB_TEXTS = dataloader.load_wikitext_calibration(num_samples=MIN_CALIB_SAMPLES)

Loading WikiText-2 dataset for calibration...


✓ Loaded 128 calibration samples from WikiText-2


## 6. AWQ (Activation-aware Weight Quantization)

**AWQ** protects important weights based on activation magnitudes:
- **Approach**: Identifies and protects salient (important) weights that correspond to large activations
- **Speed**: Moderate (faster than GPTQ, slower than RTN)
- **Accuracy**: Comparable to or better than GPTQ, especially at 4-bit
- **Use case**: Best balance of speed and accuracy for 4-bit quantization

**How it works**:
1. Analyze activation magnitudes on calibration data
2. Identify "salient" weights (those activated strongly)
3. Apply per-channel scaling to preserve these weights
4. Quantize with protection for important weights

**Key advantage**: Better preserves model performance at very low bit widths (3-4 bit)

**Reference**: [AWQ Paper](https://arxiv.org/abs/2306.00978)

In [6]:
# ============================================================================
# AWQ (ACTIVATION-AWARE WEIGHT QUANTIZATION) IMPLEMENTATION
# ============================================================================

def quantize_and_save_awq(
    model_id: str,
    bits_list: Iterable[int],
    out_root: Path = OUTPUT_ROOT,
    calib_texts: List[str] = None,
    group_size: int = 128,
    max_calib_seq_len: int = 256,
):
    """
    Quantize model using AWQ at multiple bit widths and save results.
    
    AWQ analyzes activation patterns to identify and protect important weights,
    resulting in better accuracy at low bit widths.
    
    Args:
        model_id: HuggingFace model ID or path
        bits_list: List of bit widths (AWQ typically supports 4-bit)
        out_root: Root directory for saving quantized models
        calib_texts: Calibration texts (uses CALIB_TEXTS if None)
        group_size: Group size for quantization
        max_calib_seq_len: Maximum sequence length for calibration
    """
    from awq import AutoAWQForCausalLM

    # Use default calibration texts if not provided
    calib_texts = calib_texts or CALIB_TEXTS
    
    # Load tokenizer
    tok = model_loader.load_tokenizer(model_id)

    # Quantize at each bit width
    for bits in bits_list:
        print(f"\n{'='*60}")
        print(f"Quantizing with AWQ at {bits}-bit...")
        print(f"This may take several minutes...")
        print(f"{'='*60}")
        
        # AWQ primarily supports 4-bit quantization
        if bits not in (4,):
            print(f"Warning: AWQ commonly supports 4-bit. {bits}-bit may not work.")
            raise ValueError("AutoAWQ commonly supports 4-bit weight quantization.")

        # Load model with AWQ wrapper
        model = AutoAWQForCausalLM.from_pretrained(
            model_id,
            torch_dtype=torch.float16,
            low_cpu_mem_usage=True,
            device_map="auto",
            safetensors=False
        )

        # Configure AWQ quantization
        quant_config = {
            "w_bit": bits,  # Weight bit width
            "q_group_size": group_size,  # Group size
            "zero_point": True,  # Use zero-point quantization
            "version": "GEMM",  # Use GEMM-optimized kernels
        }

        # Perform AWQ quantization (analyzes activations and quantizes)
        print("Running AWQ optimization...")
        model.quantize(
            tok,
            quant_config=quant_config,
            calib_data=calib_texts,  # Calibration texts (not tokenized)
            max_calib_samples=len(calib_texts),
            max_calib_seq_len=max_calib_seq_len,
        )

        # Save quantized model
        out_dir = out_root / f"awq_w{bits}"
        out_dir.mkdir(parents=True, exist_ok=True)
        model.save_quantized(str(out_dir))
        tok.save_pretrained(out_dir)
        
        print(f"✓ Saved AWQ {bits}-bit model to {out_dir}")

        # Clean up
        del model
        torch.cuda.empty_cache()

## 7. Run Quantization

Execute all quantization methods and save results.

**⚠️ IMPORTANT**: This cell will:
- Download the 7B parameter model (~13 GB)
- Quantize it using RTN, GPTQ, and AWQ
- Take significant time (30-60 minutes depending on hardware)
- Require ~24GB GPU memory

**Output**: Quantized models will be saved to `./quantized_models/`

In [7]:
# ============================================================================
# EXECUTE QUANTIZATION
# ============================================================================

if __name__ == "__main__":
    # Create output directory
    OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
    
    print("\n" + "="*70)
    print("STARTING QUANTIZATION PIPELINE")
    print("="*70)
    print(f"Model: {MODEL_ID}")
    print(f"Output directory: {OUTPUT_ROOT}")
    print(f"Calibration samples: {len(CALIB_TEXTS)}")
    print("="*70 + "\n")

    # AWQ Quantization (Activation-aware)
    print("\n📊 STEP 3/3: AWQ Quantization")
    quantize_and_save_awq(
        MODEL_ID, 
        bits_list=[4],  # 4-bit AWQ
        out_root=OUTPUT_ROOT
    )
    print(f"\nQuantized models saved to: {OUTPUT_ROOT}")


STARTING QUANTIZATION PIPELINE
Model: openlm-research/open_llama_7b
Output directory: quantized_models
Calibration samples: 128


📊 STEP 3/3: AWQ Quantization

Quantizing with AWQ at 4-bit...
This may take several minutes...


Fetching 10 files:   0%|          | 0/10 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

Running AWQ optimization...


AWQ: 100%|██████████| 32/32 [10:50<00:00, 20.32s/it]


Writing model shards: 0it [00:00, ?it/s]

✓ Saved AWQ 4-bit model to quantized_models/awq_w4

Quantized models saved to: quantized_models
